# Pre-Processing Meteorological Predictor Fields 
Version 14 December 2022, Selina Kiefer

### Input: netcdf- or grib-file
continuous timeseries of meteorological predictors in netcdf- or grib-format (i.e. ERA-5, various predictor fields (u10, z100, z250, z500, z850, t850, H850, u300, msl), 4 times (00, 06, 12, 18 UTC), 1950 - 2020, Oct-Apr, 60°W-60°E and 20-80°N, e.g. from https://cds.climate.copernicus.eu/#!/search?text=ERA5&type=dataset)
### Output: csv-file
continuous timeseries of meteorological predictors in csv-format

## Used software: Climate Data Operators and Python

#### Climate Data Operators (CDO) 

Taylored open-source software to perform the most-common meteorological operations efficiently (and much faster than Python). 

Up to date information about CDO: https://code.mpimet.mpg.de/projects/cdo

Reference: Schulzweida, U. (2019): "CDO User Guide". Available at: https://doi.org/10.5281/ZENODO.3539275.

#### Short introduction to CDO

The overall structure for most operations is:

cdo -operator_last_executed,optional_specifications -operator_first_executed,optional_specifcations ifile ofile

e.g. cdo -daymean -selyear,1950,1951 input_file_name output_file_name

The input file (ifile) and the output file (ofile) of one operation have to have different names. So it is best to name all files, which are not intended for further use, similarly, e.g. temp_1, temp_2, etc. and to delete them afterwards directly.

CDO does not ask when overwriting an existing file. So make sure that everything is named uniquely and correctly.

### Start with CDO

Since it is much faster than Python.

In [1]:
# Short overview of the data file's content.
!cdo sinfov ./era5_u10_180W_180E_0N_90N_1950_1978.nc

   File format : NetCDF2
    -1 : Institut Source   T Steptype Levels Num    Points Num Dtype : Parameter name
     1 : unknown  unknown  v instant       1   1     14640   1  I16  : u             
   Grid coordinates :
     1 : lonlat                   : points=14640 (240x61)
                        longitude : -180 to 178.5 by 1.5 degrees_east  circular
                         latitude : 90 to 0 by -1.5 degrees_north
   Vertical coordinates :
     1 : surface                  : levels=1
   Time coordinate :  42368 steps
     RefTime =  1900-01-01 00:00:00  Units = hours  Calendar = gregorian
  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss  YYYY-MM-DD hh:mm:ss
  1950-01-01 00:00:00  1950-01-01 06:00:00  1950-01-01 12:00:00  1950-01-01 18:00:00
  1950-01-02 00:00:00  1950-01-02 06:00:00  1950-01-02 12:00:00  1950-01-02 18:00:00
  1950-01-03 00:00:00  1950-01-03 06:00:00  1950-01-03 12:00:00  1950-01-03 18:00:00
  1950-01-04 00:00:00  1950-01-04 06:00:00  1950-01-04 12:0

In [2]:
# Detailed depiction of the data file's content. Use grib_dump for files in grib-format, 
# nc_dump for files in netcdf-format. It might be wise to use a separate terminal for this
# command since it prints all available information about the data file.
#! grib_dump ./era5_u10_60W_60E_20N_80N_1950_2020.nc
#! nc_dump ./era5_u10_60W_60E_20N_80N_1950_2020.nc

#### Spatial Preprocessing 

In [3]:
# Selection of a gridbox (sellonlatbox,°W,°E,°S,°N). Western longitudes have to be given as 
# 360°-°W). In case there is only 1 latitude or longitude to average over, select the desired
# longitude/latitude and on the second position the desired longitude/latitude+1. Otherwise 
# CDO may perform not well. 
! cdo sellonlatbox,300,60,20,80 ./era5_u10_180W_180E_0N_90N_1950_1978.nc temp_11
! cdo sellonlatbox,300,60,20,80 ./era5_u10_180W_180E_0N_90N_1979_2020.nc temp_12

cdo    sellonlatbox: Processed 620267520 values from 1 variable over 42368 timesteps [5.24s 64MB].
cdo    sellonlatbox: Processed 898368960 values from 1 variable over 61364 timesteps [7.54s 75MB].


#### Temporal Preprocessing

In [4]:
# Calculation of the daily mean (daymean). Set the time to 00 UTC (settime,00:00:00) to avoid 
# any inconveniences when reading in the data later with python.
! cdo -settime,00:00:00 -daymean temp_11 temp_21
! cdo -settime,00:00:00 -daymean temp_12 temp_22

cdo(1) daymean: Process started
cdo(1) daymean:     0%cdo    settime (Warning): Time bounds unsupported by this operator, removed!
                   1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5 5 6 6 6 6 6 6 6 6 6 6 7 7 7 7 7 7 7 7 7 7 8 8 8 8 8 8 8 8 8 8 9 9 9 9 9 9 9 9 9 910cdo(1) daymean: Processed 137272320 values from 1 variable over 52959 timesteps.
cdo    settime: Processed 34318080 values from 1 variable over 10592 timesteps [1.23s 53MB].
cdo(1) daymean: Process started
cdo(1) daymean:     0%cdo    settime (Warning): Time bounds unsupported by this operator, removed!
                   1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4 4 4 4 4 4 5 5 5 5 5 5 5 5 5 5 6 6 6 6 6 6 6 6 6 6 7 7 7 7 7 7 7 7 7 7 8 8 8 8 8 8 8 8 8 8 9 9 9 9 9 9 9 9 9 910cdo(1) daymean: Processed 198819360 values from 1 variable over 76704 timesteps.
cdo    settime: Processed 49704840 values from 1 variable over 15341 timesteps [1.7

In [5]:
# Selection of certain times, e.g. only the wintermonths (selmon).
! cdo -selmon,1,2,3,4,10,11,12 temp_21 temp_31
! cdo selmon,1,2,3,4,10,11,12 temp_22 temp_32

cdo    selmonth: Processed 19942200 values from 1 variable over 10592 timesteps [0.47s 40MB].
cdo    selmonth: Processed 28884600 values from 1 variable over 15341 timesteps [0.69s 42MB].


In [6]:
# Selection of only the relevant data according to the lead time at the beginning of each
# winter. Number of days to delete = (Days_of_Month - lead_time).
! cdo delete,day=1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,month=10 temp_31 temp_41
! cdo delete,day=1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,month=10 temp_32 temp_42

cdo    delete: Processed 18344880 values from 1 variable over 6155 timesteps [0.43s 38MB].
cdo    delete: Processed 26571240 values from 1 variable over 8915 timesteps [0.62s 40MB].


In [7]:
# Selection of only the relevant data according to the lead time at the end of each winter.
# Number of days to delete = lead_time.
! cdo delete,day=17,18,19,20,21,22,23,24,25,26,27,28,29,30,month=4 temp_41 temp_51
! cdo delete,day=17,18,19,20,21,22,23,24,25,26,27,28,29,30,month=4 temp_42 temp_52

cdo    delete: Processed 17029440 values from 1 variable over 5662 timesteps [0.41s 38MB].
cdo    delete: Processed 24666120 values from 1 variable over 8201 timesteps [0.62s 39MB].


In [8]:
# Selection of only the relevant data according to the lead time at the end of the data in case
# it does end with 31 Dec instead of 30 Apr. Number of days to
# delete = (Days_of_Month - lead_time).
! cdo delete,day=18,19,20,21,22,23,24,25,26,27,28,29,30,31,month=12,year=2020 temp_52 temp_62

cdo    delete: Processed 24620760 values from 1 variable over 7613 timesteps [0.58s 39MB].


In [9]:
# Temporal merging of two timeseries. The option "-b F64" makes sure that the two dataseries 
# can be combined without errors. 
! cdo -b F64 mergetime temp_51 temp_62 temp_7

cdo    mergetime: Processed 41650200 values from 2 variables over 12855 timesteps [0.67s 41MB].


In [10]:
# Make sure that the time is sorted correctaly (sorttimestamp) and the file is named correctly.
! cdo sorttimestamp temp_7 ./Data_in_Netcdf_Format/era5_u10_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.nc

cdo    sorttimestamp: Processed 41650200 values from 1 variable over 12855 timesteps [1.17s 361MB].


#### Convert from grib-format to netcdf-format

In [11]:
# Convert the grib-file to a netcdf-file if necessary. The python-scripts are designed to use
# netcdf-files.
#! cdo -f nc copy ofile.grib ofile.nc

#### Remove unnecessary files

In [12]:
# Remove unnecessary files which have been created by CDO since the names of the input files 
# and output files have to be unique.
! rm temp*

## Continue with Python


For a nice overview of the data, pandas dataframes are used. These are then converted directly into csv-format for storage which ensures a safe and easy data transfer between various jupyter notebooks.

#### Define the paths' and files' names 

In [13]:
# Set the needed path and file names.
PATH_defined_functions = './Defined_Functions/'

PATH_data = './Data_in_Netcdf_Format/'
ifile_data = 'era5_u10_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.nc'

PATH_output_file = './Data_in_csv_Format/'
file_name_output_file = 'era5_u10_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv'

#### Import the necessary packages and functions
Nothing needs to be changed here.

In [14]:
# Import the necessary python packages.
import numpy as np
import pandas as pd

In [15]:
# Import the necessary functions.
import sys
sys.path.insert(1,PATH_defined_functions)
from read_in_netcdf_data import *

#### Read in the data and check the file's content
Nothing needs to be changed here.

In [16]:
# Read in the data and show its header.
df_data = read_in_netcdf_data(PATH_data, ifile_data)
df_data.head()

,latitude,longitude,time,u
0,79.5,-60.0,1950-01-01,8.750118
1,79.5,-60.0,1950-01-02,11.137983
2,79.5,-60.0,1950-01-03,10.842854
3,79.5,-60.0,1950-01-04,5.043752
4,79.5,-60.0,1950-01-05,7.922222


In [17]:
# Show the end of the dataframe.
df_data.tail()

,latitude,longitude,time,u
41650195,21.0,60.0,2020-12-13,-11.602207
41650196,21.0,60.0,2020-12-14,-16.019497
41650197,21.0,60.0,2020-12-15,-14.708779
41650198,21.0,60.0,2020-12-16,-17.172759
41650199,21.0,60.0,2020-12-17,-18.960102


#### Create a minimal, useful representation of the data

In [18]:
# Extract the month of the winter to include seasonality. Needs to be done only for once for all
# ML input predictors.
df_data = df_data.set_index('time')
df_data['month'] = df_data.index.month
df_data = df_data.reset_index()

In [19]:
# Rename the variable's comlumn in case its naming is ambiguous.
df_data = df_data.rename(columns={'u':'u10'})

#### Doublecheck the representation of the data

In [20]:
# Check if everything is sorted, renamed or removed correctly.
df_data.head()

,time,latitude,longitude,u10,month
0,1950-01-01,79.5,-60.0,8.750118,1
1,1950-01-02,79.5,-60.0,11.137983,1
2,1950-01-03,79.5,-60.0,10.842854,1
3,1950-01-04,79.5,-60.0,5.043752,1
4,1950-01-05,79.5,-60.0,7.922222,1


In [21]:
# Also check if everything is sorted, renamed or removed correctly at the end of the
# dataframe.
df_data.tail()

,time,latitude,longitude,u10,month
41650195,2020-12-13,21.0,60.0,-11.602207,12
41650196,2020-12-14,21.0,60.0,-16.019497,12
41650197,2020-12-15,21.0,60.0,-14.708779,12
41650198,2020-12-16,21.0,60.0,-17.172759,12
41650199,2020-12-17,21.0,60.0,-18.960102,12


#### Save the ground truth data
Nothing needs to be changed here.

In [22]:
# Save the pandas dataframe in csv-format.
df_data.to_csv(PATH_output_file+file_name_output_file)

In [23]:
# End of Program